# BP4 Gate 5 — Decision Layer & Reporting
**Customer360 Navigator Enterprise Suite — Customer Journey Analytics**

## Why this gate looks different from BP1/BP2/BP3's own Gate 5
Master Plan Section 8's generic Gate 5 ("Decision / GenAI Layer & Reporting") bundles three real
things: a GenAI output, a decision-engine score, and reporting. For BP4 these three do not all
apply at the BP4 level, and this notebook states that honestly rather than fabricating a
substitute for the parts that belong elsewhere:

- **"GenAI"** → **Not Applicable.** This real CFPB extract carries no narrative-text column
  (Gate 1's own live-verified finding, re-confirmed at every gate since); grounded GenAI
  resolution text is explicitly BP6's scope (Master Plan Section 5.1/7) — the same standing
  scope decision already confirmed for BP1/BP2/BP3 Gate 5. No GenAI API is called here.
- **"Decision"** (a priority + intervention-risk score **combining multiple BPs' outputs**) →
  explicitly **BP7's job**. Master Plan Section 5.1/7 names BP7 (Customer Navigator Decision
  Engine) as "transparent, auditable decision rules combining BP1–BP5 outputs into a priority
  score and intervention flag." This gate does not attempt that cross-BP score — it never reaches
  outside BP4's own real, already-confirmed Gate 2/Gate 4 artifacts.
- **"Reporting"** → genuinely applies, and is what this gate builds: a transparent, fully
  auditable, deterministic per-cluster **review-priority reporting layer**, computed only from
  BP4's own real journey/cluster statistics, with grounded reason codes for every score — no
  black-box scoring, matching the "no black-box scoring" principle Section 5.1/7 states for BP7's
  own decision rules, applied here one level down at BP4's own local scope.

## What this notebook does, concretely
Reads the real, full issue-cluster summary Gold table Gate 2 built (all 37,160 real clusters, not
a sample) and computes three grounded, deterministic boolean flags per cluster — every threshold
either a real number already confirmed by a prior BP4 gate, or a real percentile computed live
from this run's own data, never an arbitrary or illustrative cutoff (financial-impact/illustrative
content is permanently banned project-wide):

1. **`recurring_flag`** — Gate 2's own real `is_recurring_cluster` field (`n_complaints_total >
   1`).
2. **`elevated_lag_flag`** — real `avg_response_lag_days` for that cluster exceeds Gate 4's own
   real bootstrap point estimate for the population mean `response_lag_days` (read live from
   `configs/bp4_customer_journey_analytics.yaml`, never re-invented — HYPER).
3. **`high_volume_flag`** — real `n_complaints_total` at or above the live-computed 90th
   percentile of `n_complaints_total` across the real cluster population this run.

`review_priority_score` is the count of triggered flags (0–3); `review_priority_tier` is a
deterministic mapping (3→HIGH, 2→MEDIUM, 1→LOW, 0→NONE). Every flagged cluster's record carries a
`reason_codes` column and a `reason_evidence` column stating the real number that triggered each
flag — grounded by construction, not asserted after the fact.

A real, live-computed tier rollup (cluster counts and real complaint-row coverage per tier) is
also written — useful downstream for BP8's Power BI layer (Master Plan Section 6's integration
architecture: "All analytical outputs → BP8 Power BI"), and cross-checked against Gate 1's own
real `journey_row_count` (every real row belongs to exactly one cluster, so the tiers' row
coverage must sum to the same real total).

## Standing rules this notebook follows
- **Execution boundary**: Claude wrote this notebook; you run it. Every flag, score, tier, and
  reason code below is computed live from your real Gate 2/Gate 4 artifacts.
- **Zero-fabrication**: no GenAI text is generated; no cross-BP decision score is fabricated in
  BP7's place. The two Not-Applicable statements above are stated honestly, not silently skipped.
- **WARP**: `configure_performance()` first; the three flags and the score/tier mapping are
  computed with vectorized Polars expressions over the real cluster population, not a per-row
  Python loop. The reason-code/evidence text formatting is a small, one-time pass over the ~37K
  real clusters (not the 1M-row event table) — the same scale of formatting pass BP3 Gate 5 used
  for its own per-row reason codes.
- **HYPER**: reuses `CLUSTER_KEY`/`BARRED_JOURNEY_COLUMNS` from
  `src/features/bp4_journey_features.py` unmodified; reuses `src/utils/bp1_config_sync.py`; reuses
  Gate 4's own already-confirmed reproducibility result (reads the Gold table directly rather than
  rebuilding the pipeline a third time).
- **Idempotent**: re-running this notebook overwrites this gate's own config block and artifacts
  in place; every other gate's block and Gate 1's front matter are preserved verbatim.
- **PROJECT_STRUCTURE_LOCKED.md rule #3**: same project-root resolver as every other notebook.

## Outputs (idempotent overwrite-in-place)
- `notebooks/bp4_customer_journey_analytics/artifacts/gate5_cluster_decision_report.csv` — one row
  per real issue-cluster (all 37,160), with its flags, score, tier, reason codes and evidence.
- `notebooks/bp4_customer_journey_analytics/artifacts/gate5_decision_layer_summary.json` — tier
  rollup, thresholds used, row-coverage cross-check, and the compliance-touchpoint statements.
- `configs/bp4_customer_journey_analytics.yaml` — Gate 5 marker block appended/overwritten.

## Prerequisites
BP4 Gates 2, 3 and 4 must all have been real-run — this notebook independently re-derives all
three prerequisites live (never trusting a later gate's own pass-through of an earlier one), and
specifically requires Gate 4's `mean_response_lag_days` bootstrap result and
`reproducibility_confirmed: True` to be present before it will compute anything.

## If a structural check below fails
It raises `AssertionError` naming the failing check. A failed
`tier_rollup_row_coverage_matches_gate1_journey_row_count` check means the real Gold table on disk
has drifted since Gate 2 ran — re-run Gate 2 for real before trusting this gate's output.


In [ ]:
# ============================================================
# SECTION 1: Project root resolution (PROJECT_STRUCTURE_LOCKED.md rule #3)
# ============================================================
import os
import sys
from pathlib import Path


def _find_project_root(marker_filename: str = "PROJECT_STRUCTURE_LOCKED.md") -> Path:
    env_override = os.environ.get("C360_PROJECT_ROOT")
    if env_override:
        candidate = Path(env_override)
        if (candidate / marker_filename).exists():
            return candidate
        raise RuntimeError(
            f"C360_PROJECT_ROOT is set to {candidate} but {marker_filename} was not found there. "
            "Fix the environment variable rather than removing this check."
        )

    start = Path.cwd()
    current = start
    for _ in range(8):
        if (current / marker_filename).exists():
            return current
        if current.parent == current:
            break
        current = current.parent

    for depth_root, dirnames, filenames in os.walk(start):
        rel_depth = len(Path(depth_root).relative_to(start).parts)
        if rel_depth > 3:
            dirnames[:] = []
            continue
        dirnames[:] = [d for d in dirnames if not d.startswith(".")]
        if marker_filename in filenames:
            return Path(depth_root)

    raise RuntimeError(
        "Could not resolve PROJECT_ROOT. Set the C360_PROJECT_ROOT environment variable to the "
        "Customer360_Navigator_Enterprise_Suite folder, or run this notebook from inside the project tree "
        "(expected at notebooks/bp4_customer_journey_analytics/)."
    )


PROJECT_ROOT = _find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
CONFIGS_DIR = PROJECT_ROOT / "configs"
DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
ARTIFACTS_DIR = PROJECT_ROOT / "notebooks" / "bp4_customer_journey_analytics" / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
BP4_CONFIG_PATH = CONFIGS_DIR / "bp4_customer_journey_analytics.yaml"
print(f"[OK] Project root resolved: {PROJECT_ROOT.name}")

# ============================================================
# SECTION 2: WARP - configure_performance() FIRST, before any heavy/BLAS-backed import.
# ============================================================
from utils.performance_setup import configure_performance  # noqa: E402

WARP_SUMMARY = configure_performance(project_root=PROJECT_ROOT, verbose=True)

# ============================================================
# SECTION 3: Heavy imports + flush-forcing print override (LESSONS_LEARNED_APPLIED.md #12).
# ============================================================
import builtins  # noqa: E402
import functools  # noqa: E402
import json  # noqa: E402
import warnings  # noqa: E402
from datetime import datetime, timezone  # noqa: E402

import polars as pl  # noqa: E402
import yaml  # noqa: E402

from features.bp4_journey_features import BARRED_JOURNEY_COLUMNS, CLUSTER_KEY  # noqa: E402
from utils.bp1_config_sync import write_gate_block  # noqa: E402

warnings.filterwarnings("ignore")
print = functools.partial(builtins.print, flush=True)

GT_SUMMARY_PATH = DATA_PROCESSED_DIR / "cfpb_issue_cluster_summary_gold.parquet"

# ============================================================
# SECTION 4: Prerequisite checks (live, independently re-derived) - Gates 2, 3 AND 4 must all have
# completed for real before this gate runs, matching this project's Three-Lines-of-Defense pattern
# where every later gate re-checks its own prerequisites rather than trusting an earlier gate's
# own pass-through of them.
# ============================================================
assert BP4_CONFIG_PATH.exists(), f"{BP4_CONFIG_PATH} does not exist - run Gate 1 first."
with open(BP4_CONFIG_PATH, "r", encoding="utf-8") as f:
    FULL_CONFIG = yaml.safe_load(f.read())

gate1_status_confirmed = "gate1_confirmed" in str(FULL_CONFIG.get("status", ""))
gate2_confirmed = (
    gate1_status_confirmed
    and FULL_CONFIG.get("journey_row_count_matches_raw") is True
    and FULL_CONFIG.get("cluster_count_matches_gate1") is True
)
gate3_confirmed = (
    gate2_confirmed
    and isinstance(FULL_CONFIG.get("candidates_correct"), int)
    and FULL_CONFIG.get("candidates_correct", 0) > 0
    and bool(FULL_CONFIG.get("champion_pipeline"))
)
gate4_confirmed = (
    gate3_confirmed
    and isinstance(FULL_CONFIG.get("mean_response_lag_days"), dict)
    and "point_estimate" in FULL_CONFIG.get("mean_response_lag_days", {})
    and FULL_CONFIG.get("reproducibility_confirmed") is True
)
assert gate4_confirmed, (
    "BP4 Gate 4 does not appear to have completed successfully (mean_response_lag_days="
    f"{FULL_CONFIG.get('mean_response_lag_days')!r}, reproducibility_confirmed="
    f"{FULL_CONFIG.get('reproducibility_confirmed')!r}). Run Gate 4 for real before Gate 5."
)
print(
    "[OK] Gate 2+3+4 prerequisites independently re-confirmed "
    f"(n_clusters={FULL_CONFIG.get('n_clusters'):,}, "
    f"champion_pipeline={FULL_CONFIG.get('champion_pipeline')!r}, "
    f"reproducibility_confirmed={FULL_CONFIG.get('reproducibility_confirmed')})."
)
assert GT_SUMMARY_PATH.exists(), f"{GT_SUMMARY_PATH} missing - run Gate 2 for real first."

GATE4_MEAN_LAG_POINT_ESTIMATE = float(FULL_CONFIG["mean_response_lag_days"]["point_estimate"])
print(
    "[OK] Gate 4's real bootstrap point estimate for population mean response_lag_days, read live "
    f"(reused as this gate's elevated-lag reference threshold, HYPER - never re-invented): "
    f"{GATE4_MEAN_LAG_POINT_ESTIMATE:.6f}"
)

# ============================================================
# SECTION 5: Scope note - what this gate builds, and what it deliberately does NOT build.
# Master Plan Section 8's generic Gate 5 ("Decision / GenAI Layer & Reporting") has three parts.
# For BP4:
#   - "GenAI" -> NOT_APPLICABLE. No narrative-text column exists in this real CFPB extract
#     (Gate 1's own live-verified finding); no GenAI API is used here - grounded GenAI resolution
#     text is explicitly BP6's scope (Master Plan Section 5.1/7), same standing decision already
#     confirmed for BP1/BP2/BP3 Gate 5.
#   - "Decision" (a priority + intervention-risk score COMBINING multiple BPs' outputs) ->
#     explicitly BP7's job (Master Plan Section 5.1/7: "Customer Navigator Decision Engine -
#     transparent, auditable decision rules combining BP1-BP5 outputs"). This gate does not
#     attempt that - it never reaches outside BP4's own real Gate 2/4 artifacts.
#   - "Reporting" -> genuinely applies, and is what this gate builds: a transparent, deterministic,
#     fully auditable per-cluster REVIEW-PRIORITY reporting layer computed ONLY from BP4's own real,
#     already-confirmed journey/cluster statistics - no black-box scoring, every flag grounded in a
#     real number from a prior BP4 gate or live-computed from the real data itself. This local flag
#     may later become one of several real inputs BP7 combines, but is not itself BP7's
#     cross-BP decision engine.
# ============================================================
print(
    "[SCOPE] This gate builds BP4's own local, deterministic reporting layer only - not BP6's "
    "GenAI layer (NOT_APPLICABLE, no narrative text / no GenAI call, scoped to BP6) and not BP7's "
    "cross-BP priority/intervention decision engine (BP7 combines BP1-BP5 outputs; this stays "
    "inside BP4's own real data)."
)

# ============================================================
# SECTION 6: Load the real, full issue-cluster summary Gold table (all real clusters, not a
# sample - Gate 4 already re-confirmed this table's reproducibility, so it is read directly here
# rather than rebuilt again, HYPER).
# ============================================================
cluster_df = pl.read_parquet(GT_SUMMARY_PATH)
n_clusters_loaded = cluster_df.height
assert n_clusters_loaded == FULL_CONFIG["n_clusters"], (
    f"[CHECK FAILED] Loaded {n_clusters_loaded:,} real clusters but Gate 1/2 recorded "
    f"{FULL_CONFIG['n_clusters']:,} - the Gold table on disk has drifted since Gate 2."
)
print(
    f"[OK] Real issue-cluster summary loaded: {n_clusters_loaded:,} clusters "
    "(matches Gate 1/2's recorded count)."
)

for barred in BARRED_JOURNEY_COLUMNS:
    assert (
        barred not in cluster_df.columns
    ), f"[CHECK FAILED] barred column '{barred}' unexpectedly present in the cluster summary table."

# ============================================================
# SECTION 7: Compute the 3 grounded, deterministic review flags - vectorized (WARP: Polars
# expressions, no per-row Python loop). Every threshold is either a real number already confirmed
# by a prior BP4 gate (Gate 4's own bootstrap point estimate) or a real percentile computed live
# from this run's own data - never a hardcoded/arbitrary cutoff, and never a financial-impact or
# illustrative figure (permanently banned project-wide).
# ============================================================
P90_THRESHOLD = float(cluster_df["n_complaints_total"].quantile(0.90, interpolation="linear"))
print(
    f"[OK] Live-computed 90th percentile of real n_complaints_total across all {n_clusters_loaded:,} "
    f"clusters (this run's own high-volume reference threshold, never hardcoded): {P90_THRESHOLD:.4f}"
)

cluster_df = cluster_df.with_columns(
    [
        pl.col("is_recurring_cluster").alias("recurring_flag"),
        (pl.col("avg_response_lag_days") > GATE4_MEAN_LAG_POINT_ESTIMATE).alias("elevated_lag_flag"),
        (pl.col("n_complaints_total") >= P90_THRESHOLD).alias("high_volume_flag"),
    ]
)
cluster_df = cluster_df.with_columns(
    (
        pl.col("recurring_flag").cast(pl.Int8)
        + pl.col("elevated_lag_flag").cast(pl.Int8)
        + pl.col("high_volume_flag").cast(pl.Int8)
    ).alias("review_priority_score")
)
cluster_df = cluster_df.with_columns(
    pl.when(pl.col("review_priority_score") == 3)
    .then(pl.lit("HIGH"))
    .when(pl.col("review_priority_score") == 2)
    .then(pl.lit("MEDIUM"))
    .when(pl.col("review_priority_score") == 1)
    .then(pl.lit("LOW"))
    .otherwise(pl.lit("NONE"))
    .alias("review_priority_tier")
)

# ============================================================
# SECTION 8: Reason codes - one grounded, human-readable evidence string per triggered flag, built
# only from that row's own real field values (never asserted after the fact). A small, one-time
# formatting pass over the real cluster population (37K rows, not the 1M-row event table) - well
# within this project's WARP scope, matching BP3 Gate 5's own identical per-row reason-code
# formatting precedent.
# ============================================================
pdf = cluster_df.to_pandas()
reason_code_lists: list[list[str]] = []
reason_evidence_lists: list[list[str]] = []
for row in pdf.itertuples(index=False):
    codes: list[str] = []
    evidence: list[str] = []
    if row.recurring_flag:
        codes.append("RECURRING")
        evidence.append(f"is_recurring_cluster=True (n_complaints_total={int(row.n_complaints_total)} > 1)")
    if row.elevated_lag_flag:
        codes.append("ELEVATED_LAG")
        evidence.append(
            f"avg_response_lag_days={row.avg_response_lag_days:.4f} > "
            f"gate4_population_mean_point_estimate={GATE4_MEAN_LAG_POINT_ESTIMATE:.4f}"
        )
    if row.high_volume_flag:
        codes.append("HIGH_VOLUME")
        evidence.append(
            f"n_complaints_total={int(row.n_complaints_total)} >= " f"live_p90_threshold={P90_THRESHOLD:.4f}"
        )
    reason_code_lists.append(codes)
    reason_evidence_lists.append(evidence)

pdf["reason_codes"] = ["|".join(c) for c in reason_code_lists]
pdf["reason_evidence"] = ["; ".join(e) for e in reason_evidence_lists]

n_with_reason_codes = int(sum(1 for c in reason_code_lists if len(c) > 0))
grounding_failures = int(
    sum(
        1
        for i, codes in enumerate(reason_code_lists)
        if ("RECURRING" in codes) != bool(pdf.iloc[i]["recurring_flag"])
        or ("ELEVATED_LAG" in codes) != bool(pdf.iloc[i]["elevated_lag_flag"])
        or ("HIGH_VOLUME" in codes) != bool(pdf.iloc[i]["high_volume_flag"])
    )
)
print(
    f"[OK] Reason codes assembled: {n_with_reason_codes:,}/{n_clusters_loaded:,} real clusters carry "
    f"at least one grounded reason code. Grounding failures: {grounding_failures}."
)

# ============================================================
# SECTION 9: Tier-level real reporting rollup (WARP: vectorized groupby, not a Python loop) - real
# cluster counts and real row-coverage per tier, useful downstream for BP8's Power BI layer
# (Section 6 integration architecture: "All analytical outputs -> BP8 Power BI"). No
# financial-impact/illustrative figures anywhere (permanently banned project-wide).
# ============================================================
tier_rollup = (
    cluster_df.group_by("review_priority_tier")
    .agg(
        pl.len().alias("n_clusters"),
        pl.col("n_complaints_total").sum().alias("n_complaint_rows_covered"),
        pl.col("banking77_coverage_fraction").mean().alias("mean_banking77_coverage_fraction"),
    )
    .sort("review_priority_tier")
)
tier_rollup_records = tier_rollup.to_dicts()
print("\n=== REVIEW-PRIORITY TIER ROLLUP (real, live-computed) ===")
for rec in tier_rollup_records:
    print(
        f"  {rec['review_priority_tier']:>6}: {rec['n_clusters']:,} clusters, "
        f"{rec['n_complaint_rows_covered']:,} real complaint rows covered, "
        f"mean BANKING77 coverage fraction={rec['mean_banking77_coverage_fraction']:.4f}"
    )
total_rows_covered_check = int(sum(r["n_complaint_rows_covered"] for r in tier_rollup_records))

# ============================================================
# SECTION 10: Write the full real per-cluster decision/reporting records (all real clusters, not a
# sample - every real cluster gets a record, matching BP3 Gate 5's "full held-out set, not a
# sample" principle applied to BP4's own full real population).
# ============================================================
output_cols = CLUSTER_KEY + [
    "n_complaints_total",
    "first_complaint_date",
    "last_complaint_date",
    "n_active_months",
    "avg_response_lag_days",
    "banking77_coverage_fraction",
    "is_recurring_cluster",
    "recurring_flag",
    "elevated_lag_flag",
    "high_volume_flag",
    "review_priority_score",
    "review_priority_tier",
    "reason_codes",
    "reason_evidence",
]
decision_report_df = pdf[output_cols].sort_values("review_priority_score", ascending=False)
assert len(decision_report_df) == n_clusters_loaded, (
    f"[CHECK FAILED] Decision-report row count ({len(decision_report_df)}) does not match the real "
    f"cluster population ({n_clusters_loaded})."
)

records_path = ARTIFACTS_DIR / "gate5_cluster_decision_report.csv"
decision_report_df.to_csv(records_path, index=False)
print(
    f"\n[SAVED] {records_path.relative_to(PROJECT_ROOT)} ({len(decision_report_df):,} real per-cluster "
    "reporting records, one row per real issue-cluster)"
)

# ============================================================
# SECTION 11: Compliance touchpoints (Master Plan Section 8's Gate 5 row) - both stated honestly,
# neither silently skipped, matching the standing scope decision already confirmed for
# BP1/BP2/BP3 Gate 5.
# ============================================================
compliance_touchpoint = {
    "udaap_language_review": (
        "Not Applicable to BP4 Gate 5 - this gate generates no GenAI or customer-facing text; "
        "review_priority_score/tier and reason_codes/reason_evidence are deterministic outputs of "
        "a transparent rule evaluated over real, already-confirmed BP4 statistics. Real "
        "GenAI-drafted customer-facing text (subject to UDAAP review) is scoped to BP6 per the "
        "Master Plan, the same standing scope decision confirmed for BP1/BP2/BP3 Gate 5."
    ),
    "nist_ai_rmf_measure_manage": (
        "Not Applicable to BP4 Gate 5 for the same reason - no GenAI output is produced here. "
        "Applies at BP6."
    ),
    "bp7_decision_engine_boundary": (
        "review_priority_score/tier is a BP4-local reporting flag built only from BP4's own real "
        "Gate 2/4 statistics. It is not the cross-BP priority + intervention-risk decision engine "
        "Master Plan Section 5.1/7 scopes to BP7 (which combines BP1-BP5 outputs with its own "
        "reason codes and thresholds) - this gate's output may become one real input BP7 later "
        "combines, but is never presented as BP7's own decision."
    ),
    "genai_api_used": False,
    "scope_decision_confirmed_by_user_utc": "2026-09-22",
}
print("\n=== COMPLIANCE TOUCHPOINTS ===")
for key, value in compliance_touchpoint.items():
    print(f"  {key}: {value}")

# ============================================================
# SECTION 12: Write summary (idempotent overwrite-in-place).
# ============================================================
summary = {
    "bp_id": "bp4",
    "gate": 5,
    "n_clusters": n_clusters_loaded,
    "n_with_reason_codes": n_with_reason_codes,
    "grounding_failures": grounding_failures,
    "gate4_population_mean_lag_reference": GATE4_MEAN_LAG_POINT_ESTIMATE,
    "live_p90_high_volume_threshold": P90_THRESHOLD,
    "tier_rollup": tier_rollup_records,
    "total_rows_covered_check": total_rows_covered_check,
    "total_rows_covered_matches_gate1": total_rows_covered_check == FULL_CONFIG.get("journey_row_count"),
    "compliance_touchpoint": compliance_touchpoint,
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
}
summary_path = ARTIFACTS_DIR / "gate5_decision_layer_summary.json"
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)
print(f"[SAVED] {summary_path.relative_to(PROJECT_ROOT)}")

# ============================================================
# SECTION 13: Write the Gate 5 config block (marker-based, order-independent - reuses
# bp1_config_sync.py unmodified, sixth BP4 gate to do so).
# ============================================================
gate5_marker = "# --- Gate 5 (Decision Layer & Reporting) results (appended, idempotent overwrite) ---"
gate5_block_lines = [
    f"n_clusters_reported: {n_clusters_loaded}",
    f"n_with_reason_codes: {n_with_reason_codes}",
    f"gate4_population_mean_lag_reference: {GATE4_MEAN_LAG_POINT_ESTIMATE}",
    f"live_p90_high_volume_threshold: {P90_THRESHOLD}",
    "tier_rollup:",
]
for rec in tier_rollup_records:
    gate5_block_lines.append(f'  {rec["review_priority_tier"]}:')
    gate5_block_lines.append(f"    n_clusters: {rec['n_clusters']}")
    gate5_block_lines.append(f"    n_complaint_rows_covered: {rec['n_complaint_rows_covered']}")
gate5_block_lines.extend(
    [
        f"total_rows_covered_matches_gate1: {summary['total_rows_covered_matches_gate1']}",
        "genai_api_used: false",
        f'generated_at_utc: "{summary["generated_at_utc"]}"',
    ]
)
write_gate_block(BP4_CONFIG_PATH, gate5_marker, gate5_block_lines)
print(f"[SAVED] {BP4_CONFIG_PATH.relative_to(PROJECT_ROOT)} (gate5 block)")

with open(BP4_CONFIG_PATH, "r", encoding="utf-8") as f:
    _post_write_config_text = f.read()
gate5_block_actually_written = gate5_marker in _post_write_config_text

# ============================================================
# SECTION 14: Structural integrity checks - raise AssertionError, never silently pass
# ============================================================
checks = {
    "gate2_prerequisite_confirmed": gate2_confirmed,
    "gate3_prerequisite_confirmed": gate3_confirmed,
    "gate4_prerequisite_confirmed": gate4_confirmed,
    "cluster_population_loaded_matches_gate1_gate2": n_clusters_loaded == FULL_CONFIG["n_clusters"],
    "no_barred_column_in_cluster_report": all(b not in cluster_df.columns for b in BARRED_JOURNEY_COLUMNS),
    "review_priority_score_in_valid_range": pdf["review_priority_score"].between(0, 3).all(),
    "review_priority_tier_values_valid": set(pdf["review_priority_tier"].unique()).issubset(
        {"HIGH", "MEDIUM", "LOW", "NONE"}
    ),
    "decision_report_row_count_matches_cluster_population": len(decision_report_df) == n_clusters_loaded,
    "all_reason_codes_grounded": grounding_failures == 0,
    "tier_rollup_row_coverage_matches_gate1_journey_row_count": summary["total_rows_covered_matches_gate1"],
    "compliance_touchpoint_documented": bool(compliance_touchpoint),
    "decision_report_csv_written": records_path.exists(),
    "summary_json_written": summary_path.exists(),
    "config_gate5_block_written": gate5_block_actually_written,
}

print("\n=== INTEGRITY CHECKS ===")
for check_name, passed in checks.items():
    result_label = "[PASS]" if passed else "[FAIL]"
    print(f"{result_label} {check_name}")
    assert passed, f"[CHECK FAILED] {check_name}"

print(
    f"\n[ALL CHECKS PASSED] BP4 Gate 5 complete - {n_clusters_loaded:,} real per-cluster reporting "
    f"records written ({n_with_reason_codes:,} with grounded reason codes). Tier rollup: "
    + ", ".join(f"{r['review_priority_tier']}={r['n_clusters']:,}" for r in tier_rollup_records)
    + ". No GenAI API used (BP6-scoped); this gate's local flag is not BP7's cross-BP decision "
    "engine. Proceed to BP4 Gate 6 (Productization, Monitoring & Governance) next."
)
